# PARTE 1 - Recolección de noticias

## 1.1 Configuración inicial 

**Fecha de descarga:** 18 de octubre de 2026.

**Medio elegido:** A (La República).

**URLs base:**

https://larepublica.pe/ultimas-noticias

https://larepublica.pe/rss/home.xml

https://larepublica.pe/rss/sociedad.xml

https://larepublica.pe/rss/latinos-en-usa.xml

https://larepublica.pe/rss/politica.xml

https://larepublica.pe/rss/opinion.xml

https://larepublica.pe/rss/mundo.xml

https://larepublica.pe/rss/mastergrama.xml

https://larepublica.pe/rss/espectaculos.xml

https://larepublica.pe/rss/economia.xml

https://larepublica.pe/rss/deportes.xml

https://larepublica.pe/rss/cine-series.xml

https://larepublica.pe/rss/carlincatura.xml

https://larepublica.pe/rss/ciencia.xml

https://larepublica.pe/rss/servicios.xml

https://larepublica.pe/rss/horoscopo.xml


## 1.2 Configuración de user agent y revisión de robots.txt

In [38]:
import html
import json
import random
import re
import time
import unicodedata
from datetime import datetime
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.parse import unquote, urljoin, urlsplit
from zoneinfo import ZoneInfo

import pandas as pd
import requests
from bs4 import BeautifulSoup

MEDIO = "La República"
BASE = "https://larepublica.pe"
URL_ULTIMAS = f"{BASE}/ultimas-noticias"
META = 350

USER_AGENT = (
    "DiplomadoPUCP-PythonIntermedio-Tarea1/1.0 "
    "(scraper academico; +https://github.com/dgarcesg/diplomado_python_intermedio)"
)
PAUSA_MIN, PAUSA_MAX = 1.0, 2.0  # segundos entre peticiones
MAX_REINTENTOS = 1
TIMEOUT = 20
LIMA = ZoneInfo("America/Lima")

# Cada ejecución guarda sus respuestas crudas en una carpeta propia
EJECUCION = datetime.now(LIMA).strftime("%Y%m%d_%H%M%S")
DIR_RAW = Path("datos") / "raw" / f"larepublica_{EJECUCION}"
DIR_RAW.mkdir(parents=True, exist_ok=True)

sesion = requests.Session()
sesion.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "es-PE,es;q=0.9"})
print("Respaldo crudo en:", DIR_RAW)

Respaldo crudo en: datos\raw\larepublica_20261018_225700


In [39]:
BITACORA = []  # una fila por intento de petición


def pedir(url, nombre_crudo, respetar_robots=True):
    """GET cortés. Guarda la respuesta cruda en DIR_RAW y la devuelve, o None si falla."""
    if respetar_robots and not permitido(url):
        BITACORA.append({"url": url, "intento": 0, "resultado": "prohibido por robots.txt"})
        print(f"  robots.txt prohíbe {url}: no se pide")
        return None

    for intento in range(1, MAX_REINTENTOS + 2):
        time.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX) * intento)
        fila = {"momento": datetime.now(LIMA).isoformat(timespec="seconds"), "url": url, "intento": intento}
        try:
            respuesta = sesion.get(url, timeout=TIMEOUT)
        except requests.RequestException as error:
            BITACORA.append({**fila, "resultado": f"error de red: {type(error).__name__}"})
            continue

        fila["estado"] = respuesta.status_code
        if respuesta.status_code == 200:
            archivo = DIR_RAW / nombre_crudo
            archivo.write_bytes(respuesta.content)
            BITACORA.append({**fila, "resultado": "ok", "bytes": len(respuesta.content), "archivo_crudo": archivo.name})
            return respuesta
        if respuesta.status_code in (401, 403):
            BITACORA.append({**fila, "resultado": "bloqueado"})
            return None
        BITACORA.append({**fila, "resultado": "error HTTP"})
        if respuesta.status_code != 429 and respuesta.status_code < 500:
            return None  # 404 y similares: reintentar no ayuda
    return None

In [40]:
def patron_a_regex(patron):
    """Convierte un patrón de robots.txt (con * y $) en una expresión regular."""
    regex = re.escape(patron).replace(r"\*", ".*")
    if regex.endswith(r"\$"):
        regex = regex[:-2] + "$"
    return re.compile(regex)


def leer_robots(texto, user_agent):
    """Devuelve (reglas, crawl_delay, sitemaps) del grupo que aplica a user_agent."""
    grupos, grupo, sitemaps = [], None, []
    leyendo_agentes = False
    for linea in texto.splitlines():
        linea = linea.split("#", 1)[0].strip()
        if ":" not in linea:
            continue
        campo, valor = (parte.strip() for parte in linea.split(":", 1))
        campo = campo.lower()
        if campo == "user-agent":
            # Varias líneas User-agent seguidas comparten un mismo grupo de reglas
            if not leyendo_agentes:
                grupo = {"agentes": [], "reglas": [], "crawl_delay": None}
                grupos.append(grupo)
            grupo["agentes"].append(valor.lower())
            leyendo_agentes = True
        elif campo == "sitemap":
            sitemaps.append(valor)
        elif grupo is not None:
            leyendo_agentes = False
            if campo in ("allow", "disallow") and valor:
                grupo["reglas"].append((campo == "allow", valor, patron_a_regex(valor)))
            elif campo == "crawl-delay":
                grupo["crawl_delay"] = float(valor)

    nombre = user_agent.split("/")[0].lower()
    elegidos = [g for g in grupos if nombre in g["agentes"]] or [g for g in grupos if "*" in g["agentes"]]
    reglas = [regla for g in elegidos for regla in g["reglas"]]
    demora = max((g["crawl_delay"] for g in elegidos if g["crawl_delay"] is not None), default=None)
    return reglas, demora, sitemaps


def permitido(url):
    """True si robots.txt permite pedir esta URL."""
    partes = urlsplit(url)
    ruta = unquote(partes.path or "/") + (f"?{partes.query}" if partes.query else "")
    mejor_largo, permite = -1, True
    for es_allow, patron, regex in REGLAS:
        if regex.match(ruta) and (len(patron), es_allow) > (mejor_largo, permite):
            mejor_largo, permite = len(patron), es_allow
    return permite


respuesta = pedir(f"{BASE}/robots.txt", "robots.txt", respetar_robots=False)
REGLAS, CRAWL_DELAY, SITEMAPS = leer_robots(respuesta.text, USER_AGENT)
if CRAWL_DELAY:  # si el medio pide una demora mayor, se respeta
    PAUSA_MIN = max(PAUSA_MIN, CRAWL_DELAY)
    PAUSA_MAX = max(PAUSA_MAX, PAUSA_MIN)
print(f"{len(REGLAS)} reglas · Crawl-delay: {CRAWL_DELAY} · {len(SITEMAPS)} líneas Sitemap\n")

for prueba in [URL_ULTIMAS, f"{URL_ULTIMAS}/pag/2", f"{BASE}/rss/home.xml", f"{BASE}/politica", f"{BASE}/buscador?q=peru"]:
    print(f"{'permitido' if permitido(prueba) else 'PROHIBIDO':>10}  {prueba}")

27 reglas · Crawl-delay: None · 45 líneas Sitemap

 permitido  https://larepublica.pe/ultimas-noticias
 PROHIBIDO  https://larepublica.pe/ultimas-noticias/pag/2
 permitido  https://larepublica.pe/rss/home.xml
 permitido  https://larepublica.pe/politica
 PROHIBIDO  https://larepublica.pe/buscador?q=peru


## 2.1 Extracción últimas noticias

In [41]:
PATRON_NOTICIA = re.compile(r"/\d{4}/\d{2}/\d{2}/")  # las noticias llevan su fecha en la URL


def limpiar_texto(texto):
    """Quita etiquetas HTML y espacios de sobra."""
    if not texto:
        return ""
    if "<" in texto:
        texto = BeautifulSoup(texto, "html.parser").get_text(" ")
    return " ".join(html.unescape(texto).split())


def normalizar_url(url):
    """URL absoluta, sin parámetros ni fragmentos."""
    return urlsplit(urljoin(BASE, url.strip()))._replace(query="", fragment="").geturl()


def seccion_desde_url(url):
    return urlsplit(url).path.strip("/").split("/")[0]


def json_de_pagina(texto_html):
    """pageProps del JSON __NEXT_DATA__ de la página, o None si no está."""
    script = BeautifulSoup(texto_html, "lxml").find("script", id="__NEXT_DATA__")
    return json.loads(script.string)["props"]["pageProps"] if script else None


def noticias_en_json(nodo):
    """Recorre el JSON y devuelve cada objeto que tenga forma de noticia."""
    if isinstance(nodo, dict):
        slug = nodo.get("slug")
        if nodo.get("title") and nodo.get("date") and isinstance(slug, str) and PATRON_NOTICIA.search(slug):
            yield nodo
        for valor in nodo.values():
            yield from noticias_en_json(valor)
    elif isinstance(nodo, list):
        for valor in nodo:
            yield from noticias_en_json(valor)


def registro_json(articulo, fuente):
    """Convierte una noticia del JSON de la página en una fila."""
    datos = articulo.get("data") or {}
    url = normalizar_url(articulo["slug"])
    categorias = datos.get("categories") or []
    principal = next((c for c in categorias if c.get("primary")), categorias[0] if categorias else None)
    return {
        "titular": limpiar_texto(articulo["title"]),
        "bajada": limpiar_texto(datos.get("teaser")),
        "medio": MEDIO,
        "seccion_original": principal["name"] if principal else "",
        "url": url,
        "fecha": datetime.fromisoformat(articulo["date"]).replace(tzinfo=LIMA).isoformat(),
        "fuente": fuente,
    }


def unicas(registros):
    return len({r["url"] for r in registros})


registros = []
respuesta = pedir(URL_ULTIMAS, "ultimas-noticias.html")
props = json_de_pagina(respuesta.text) if respuesta else None
if props is None:
    print("Sin acceso al HTML de /ultimas-noticias: se continúa con los RSS oficiales.")
else:
    ultimas = [registro_json(a, "html_ultimas_noticias") for a in noticias_en_json(props.get("data_lastNews"))]
    registros += ultimas
    print(f"/ultimas-noticias: {len(ultimas)} noticias")

print(f"¿robots.txt permite paginar (/ultimas-noticias/pag/2)? {permitido(URL_ULTIMAS + '/pag/2')}")

/ultimas-noticias: 30 noticias
¿robots.txt permite paginar (/ultimas-noticias/pag/2)? False


## 2.2 Extracción RSS

In [42]:
def registros_rss(xml, fuente):
    """Convierte los <item> de un feed RSS en filas."""
    filas = []
    for item in BeautifulSoup(xml, "xml").find_all("item"):
        enlace = item.find("link", recursive=False)
        if enlace is None:
            continue
        url = normalizar_url(enlace.get_text(strip=True))
        titulo = item.find("title", recursive=False)
        descripcion = item.find("description", recursive=False)
        categoria = item.find("category", recursive=False)
        fecha = item.find("pubDate", recursive=False)
        filas.append({
            "titular": limpiar_texto(titulo.get_text() if titulo else ""),
            "bajada": limpiar_texto(descripcion.get_text() if descripcion else ""),
            "medio": MEDIO,
            "seccion_original": categoria.get_text(strip=True) if categoria else "",
            "url": url,
            "fecha": parsedate_to_datetime(fecha.get_text(strip=True)).astimezone(LIMA).isoformat() if fecha else "",
            "fuente": fuente,
        })
    return filas


feeds = sorted((u for u in SITEMAPS if "/rss/" in u), key=lambda u: not u.endswith("/home.xml"))
print(f"{len(feeds)} RSS oficiales declarados en robots.txt\n")

for feed in feeds:
    if unicas(registros) >= META:
        break
    respuesta = pedir(feed, f"rss_{Path(urlsplit(feed).path).stem}.xml")
    if respuesta is None:
        continue
    nuevas = registros_rss(respuesta.content, "rss")
    registros += nuevas
    print(f"{feed}: {len(nuevas)} ítems · únicas acumuladas: {unicas(registros)}")

15 RSS oficiales declarados en robots.txt

https://larepublica.pe/rss/home.xml: 196 ítems · únicas acumuladas: 196
https://larepublica.pe/rss/sociedad.xml: 42 ítems · únicas acumuladas: 196
https://larepublica.pe/rss/latinos-en-usa.xml: 8 ítems · únicas acumuladas: 196
https://larepublica.pe/rss/politica.xml: 23 ítems · únicas acumuladas: 196
https://larepublica.pe/rss/opinion.xml: 13 ítems · únicas acumuladas: 196
https://larepublica.pe/rss/mundo.xml: 22 ítems · únicas acumuladas: 197
https://larepublica.pe/rss/mastergrama.xml: 2 ítems · únicas acumuladas: 197
https://larepublica.pe/rss/espectaculos.xml: 32 ítems · únicas acumuladas: 197
https://larepublica.pe/rss/economia.xml: 6 ítems · únicas acumuladas: 197
https://larepublica.pe/rss/deportes.xml: 14 ítems · únicas acumuladas: 197
https://larepublica.pe/rss/cine-series.xml: 5 ítems · únicas acumuladas: 197
https://larepublica.pe/rss/carlincatura.xml: 2 ítems · únicas acumuladas: 197
https://larepublica.pe/rss/ciencia.xml: 10 ítems 

## 2.3 Extracción portadas de sección

In [43]:
SECCIONES = [
    "politica", "economia", "sociedad", "mundo", "deportes", "espectaculos", "ciencia",
    "cultural", "judicialidad", "verificador", "opinion", "columnistas", "latinos-en-usa",
    "cine-series", "cultura-asiatica", "entretenimiento", "servicios", "tecnologia",
    "mundo/argentina", "mundo/venezuela", "mundo/mexico",
]

for seccion in SECCIONES:
    if unicas(registros) >= META:
        break
    respuesta = pedir(f"{BASE}/{seccion}", f"seccion_{seccion.replace('/', '_')}.html")
    if respuesta is None and BITACORA[-1]["resultado"] == "bloqueado":
        print(f"/{seccion}: el medio bloquea el HTML (403). Se detiene y se usa lo obtenido por RSS.")
        break
    props = json_de_pagina(respuesta.text) if respuesta else None
    if props is None:
        print(f"/{seccion}: sin datos")
        continue
    nuevas = [registro_json(a, "html_seccion") for a in noticias_en_json(props)]
    registros += nuevas
    print(f"/{seccion}: {len(nuevas)} noticias · únicas acumuladas: {unicas(registros)}")

/politica: 12 noticias · únicas acumuladas: 197
/economia: 12 noticias · únicas acumuladas: 203
/sociedad: 12 noticias · únicas acumuladas: 203
/mundo: 12 noticias · únicas acumuladas: 203
/deportes: 12 noticias · únicas acumuladas: 203
/espectaculos: 12 noticias · únicas acumuladas: 203
/ciencia: 12 noticias · únicas acumuladas: 205
/cultural: 12 noticias · únicas acumuladas: 210
/judicialidad: 36 noticias · únicas acumuladas: 240
/verificador: 25 noticias · únicas acumuladas: 261
/opinion: 75 noticias · únicas acumuladas: 312
/columnistas: 52 noticias · únicas acumuladas: 312
/latinos-en-usa: 12 noticias · únicas acumuladas: 316
/cine-series: 12 noticias · únicas acumuladas: 323
/cultura-asiatica: 12 noticias · únicas acumuladas: 333
/entretenimiento: 12 noticias · únicas acumuladas: 341
/servicios: 12 noticias · únicas acumuladas: 351


## 3. Limpieza y normalización

### 3.1 Eliminación de duplicados y titulares vacíos

In [44]:
COLUMNAS = ["titular", "bajada", "medio", "seccion_original", "url", "fecha"]

crudo = pd.DataFrame(registros)
descartes = []  # (motivo, filas descartadas) para el reporte de 6.3

sin_titular = crudo["titular"].fillna("").str.strip().eq("")
descartes.append(("titular vacío", crudo[sin_titular]))
con_titular = crudo[~sin_titular]

repetida = con_titular.duplicated("url", keep="first")
descartes.append(("URL duplicada", con_titular[repetida]))
noticias = con_titular[~repetida].sort_values("fecha", ascending=False, ignore_index=True)

# Secciones vacías: el nombre que el medio da a ese tramo de URL en el resto de notas
tramo_crudo = crudo["url"].map(seccion_desde_url)
nombre_del_tramo = (
    crudo[crudo["seccion_original"].ne("")]
    .groupby(tramo_crudo)["seccion_original"]
    .agg(lambda nombres: nombres.mode()[0])
)
sin_seccion = noticias["seccion_original"].eq("")
tramo = noticias.loc[sin_seccion, "url"].map(seccion_desde_url)
noticias.loc[sin_seccion, "seccion_original"] = tramo.map(nombre_del_tramo).fillna(tramo)

print(f"Filas extraídas: {len(crudo)} · noticias únicas con titular: {len(noticias)}")
assert len(noticias) >= META, f"Solo hay {len(noticias)} noticias; faltan {META - len(noticias)}"
print(f"Secciones completadas desde la URL: {sin_seccion.sum()}")
print(f"Bajadas vacías: {noticias['bajada'].eq('').sum()}")
print(f"Rango de fechas: {noticias['fecha'].min()} → {noticias['fecha'].max()}\n")
print(noticias["fuente"].value_counts().to_string())

Filas extraídas: 753 · noticias únicas con titular: 351
Secciones completadas desde la URL: 41
Bajadas vacías: 2
Rango de fechas: 2023-02-18T19:25:51-05:00 → 2026-10-18T22:37:33-05:00

fuente
rss                      167
html_seccion             154
html_ultimas_noticias     30


### 3.2 Mapeo a las categorías politica, economia, deportes, ciencia_tecnología

In [45]:
EQUIVALENCIAS = {
    "politica": "politica",
    "economia": "economia",
    "precio del dolar": "economia",  # subsección de Economía (/economia/precio-del-dolar/)
    "deportes": "deportes",
    "ciencia": "ciencia_tecnologia",
    "tecnologia": "ciencia_tecnologia",
}


def normalizar(texto):
    """Minúsculas y sin tildes: 'Política' -> 'politica'."""
    sin_tildes = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sin_tildes.lower().strip()


noticias["etiqueta"] = noticias["seccion_original"].map(normalizar).map(EQUIVALENCIAS)

etiquetadas = noticias["etiqueta"].notna()
print(f"Con etiqueta: {etiquetadas.sum()} · sin etiqueta: {(~etiquetadas).sum()}\n")
print(noticias["etiqueta"].value_counts().to_string())

# Tabla de equivalencias: cada sección encontrada, su etiqueta y cuántas noticias tiene
tabla_equivalencias = (
    noticias.groupby("seccion_original")
    .agg(etiqueta=("etiqueta", "first"), noticias=("url", "size"))
    .sort_values(["etiqueta", "noticias"], ascending=[True, False], na_position="last")
    .fillna({"etiqueta": "(sin etiqueta)"})
    .reset_index()
)
tabla_equivalencias

Con etiqueta: 62 · sin etiqueta: 289

etiqueta
politica              24
deportes              14
ciencia_tecnologia    12
economia              12


,seccion_original,etiqueta,noticias
0,Ciencia,ciencia_tecnologia,12
1,Deportes,deportes,14
2,Economía,economia,9
3,Precio del dólar,economia,3
4,Política,politica,24
5,Opinión,(sin etiqueta),63
6,Sociedad,(sin etiqueta),42
7,Espectáculos,(sin etiqueta),32
8,Verificador,(sin etiqueta),25
9,Mundo,(sin etiqueta),23


#### Tabla de equivalencias

| seccion_original | etiqueta |
|---|---|
| Política | `politica` |
| Economía | `economia` |
| Precio del dólar | `economia` |
| Deportes | `deportes` |
| Ciencia | `ciencia_tecnologia` |
| Tecnología | `ciencia_tecnologia` |
| Opinión | (sin etiqueta) |
| Carlincatura, Molina, Heduardicidios | (sin etiqueta) |
| Mundo | (sin etiqueta) |
| Verificador | (sin etiqueta)
| Latinos en USA | (sin etiqueta) |
| sullana, piura, tumbes, Lambayeque, Del Santa, La Libertad | (sin etiqueta) |
| Espectáculos, Entretenimiento, Cine y series, Cultura Asiática, Cultural | (sin etiqueta) |
| Horóscopo, Mastergrama | (sin etiqueta) |

### 3.3 Reporte de descartes

In [46]:
total_descartadas = sum(len(filas) for _, filas in descartes)
assert len(crudo) == total_descartadas + len(noticias), "El reporte no cuadra con las filas extraídas"

reporte = pd.DataFrame(
    [("Filas extraídas", len(crudo))]
    + [(f"Descartadas: {motivo}", len(filas)) for motivo, filas in descartes]
    + [
        ("Total descartadas", total_descartadas),
        ("Noticias limpias", len(noticias)),
        ("· con etiqueta", int(noticias["etiqueta"].notna().sum())),
        ("· sin etiqueta (se conservan)", int(noticias["etiqueta"].isna().sum())),
    ],
    columns=["concepto", "noticias"],
)

duplicadas = dict(descartes)["URL duplicada"]
print("URL duplicadas según la fuente que las repitió:", duplicadas["fuente"].value_counts().to_dict())
reporte

URL duplicadas según la fuente que las repitió: {'rss': 212, 'html_seccion': 190}


,concepto,noticias
0,Filas extraídas,753
1,Descartadas: titular vacío,0
2,Descartadas: URL duplicada,402
3,Total descartadas,402
4,Noticias limpias,351
5,· con etiqueta,62
6,· sin etiqueta (se conservan),289


## 4. Análisis exploratorio

In [50]:
import plotly.express as px

eda = noticias[COLUMNAS + ["etiqueta"]].copy()  # tabla limpia de la sección 6
eda["fecha"] = pd.to_datetime(eda["fecha"], errors="coerce")
total = len(eda)

AZUL = "#2a78d6"   # la serie que se analiza
GRIS = "#c3c2b7"   # lo que queda fuera del análisis
TINTA = "#52514e"  # textos secundarios
REJILLA = "#e1e0d9"
FONDO = "#fcfcfb"


def estilo(fig, subtitulo, horizontal=False):
    """Aspecto común: fondo claro, rejilla tenue, título a la izquierda y subtítulo."""
    fig.update_layout(
        template="plotly_white",
        title=dict(x=0, xref="paper", subtitle=dict(text=subtitulo, font=dict(color=TINTA, size=13))),
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=13, color="#0b0b0b"),
        plot_bgcolor=FONDO,
        paper_bgcolor=FONDO,
        showlegend=False,
        bargap=0.25,
        margin=dict(t=100, l=10, r=40, b=50),
    )
    # Rejilla solo en el eje de los valores
    fig.update_xaxes(showgrid=horizontal, gridcolor=REJILLA, linecolor=GRIS, zeroline=False)
    fig.update_yaxes(showgrid=not horizontal, gridcolor=REJILLA, linecolor=GRIS, zeroline=False)
    fig.update_traces(marker_cornerradius=4, selector=dict(type="bar"))
    return fig


print(f"{total} noticias · medios: {', '.join(eda['medio'].unique())}")

351 noticias · medios: La República


### 4.1 Distribución de noticias por categoría

In [48]:
NOMBRES = {
    "politica": "Política",
    "economia": "Economía",
    "deportes": "Deportes",
    "ciencia_tecnologia": "Ciencia y tecnología",
}
eda["categoria"] = eda["etiqueta"].map(NOMBRES).fillna("Sin etiqueta")

por_categoria = eda["categoria"].value_counts().rename_axis("categoria").reset_index(name="noticias")
etiquetadas = eda["etiqueta"].notna().sum()
lider = por_categoria[por_categoria["categoria"] != "Sin etiqueta"].iloc[0]

fig = px.bar(
    por_categoria,
    x="noticias",
    y="categoria",
    orientation="h",
    text="noticias",
    color="categoria",
    color_discrete_map={c: GRIS if c == "Sin etiqueta" else AZUL for c in por_categoria["categoria"]},
    labels={"noticias": "Noticias", "categoria": ""},
    title=(
        f"Solo {etiquetadas} de {total} noticias ({etiquetadas / total:.0%}) caen en las 4 categorías; "
        f"{lider['categoria']} lidera con {lider['noticias']}"
    ),
)
fig.update_yaxes(categoryorder="total ascending")
fig.update_traces(textposition="outside", cliponaxis=False, hovertemplate="%{y}: %{x} noticias<extra></extra>")
estilo(fig, f"Noticias por categoría temática · n = {total}", horizontal=True).show()

### 4.2 Distribución de noticias por rango de fechas

In [51]:
RANGOS = ["Menos de 48 h", "2 a 7 días", "8 a 30 días", "1 a 12 meses", "Más de 1 año"]
referencia = eda["fecha"].max()
dias = (referencia - eda["fecha"]).dt.total_seconds() / 86_400
eda["antiguedad"] = pd.cut(dias, bins=[-0.001, 2, 7, 30, 365, float("inf")], labels=RANGOS)

por_rango = eda["antiguedad"].value_counts(sort=False).rename_axis("antiguedad").reset_index(name="noticias")
recientes = por_rango.loc[0, "noticias"] / total
mas_de_un_mes = por_rango.loc[por_rango["antiguedad"].isin(RANGOS[3:]), "noticias"].sum()

fig = px.bar(
    por_rango,
    x="antiguedad",
    y="noticias",
    text="noticias",
    color_discrete_sequence=[AZUL],
    labels={"antiguedad": "", "noticias": "Noticias"},
    title=f"El {recientes:.0%} de las noticias tiene menos de 48 horas, pero {mas_de_un_mes} superan el mes de antigüedad",
)
fig.update_traces(textposition="outside", cliponaxis=False, hovertemplate="%{x}: %{y} noticias<extra></extra>")
estilo(fig, f"Noticias por antigüedad, contada desde la más reciente ({referencia:%d-%m-%Y %H:%M}) · n = {total}").show()

### 4.3 Histograma de longitud de caracteres de los titulares

In [52]:
ANCHO_BARRA = 5  # caracteres por barra; la sección 10 usa el mismo ancho
eda["largo_titular"] = eda["titular"].str.len()
q1, mediana, q3 = eda["largo_titular"].quantile([0.25, 0.5, 0.75]).round().astype(int)

fig = px.histogram(
    eda,
    x="largo_titular",
    color_discrete_sequence=[AZUL],
    labels={"largo_titular": "Caracteres del titular"},
    title=f"La mitad de los titulares mide entre {q1} y {q3} caracteres (mediana: {mediana})",
)
fig.add_vrect(x0=q1, x1=q3, fillcolor=AZUL, opacity=0.08, line_width=0, layer="below")
fig.add_vline(x=mediana, line_width=2, line_color=TINTA, annotation_text=f"Mediana: {mediana}", annotation_position="top right")
fig.update_traces(xbins=dict(start=0, size=ANCHO_BARRA), hovertemplate="%{x} caracteres: %{y} titulares<extra></extra>")
fig.update_yaxes(title="Noticias")
estilo(fig, f"Longitud del titular en caracteres · franja: 50% central · n = {total}")
fig.update_layout(bargap=0.08)
fig.show()

## 5. Validación

In [53]:
VER = ["largo_titular", "titular", "seccion_original"]
mas_largos = eda.nlargest(5, "largo_titular")[VER]
mas_cortos = eda.nsmallest(5, "largo_titular")[VER]

display(mas_largos.style.set_caption("Los 5 titulares más largos").hide(axis="index"))
display(mas_cortos.style.set_caption("Los 5 titulares más cortos").hide(axis="index"))

largo_titular,titular,seccion_original
177,"De sufrir bullying escolar a ganar la final de Yo Soy: conoce la dura historia de Nikólas Bujahico, artista de Huaral, que triunfó imitando al cantante argentino Adrián Barilari",Entretenimiento
164,"En 1986, la explosión del reactor de Chernóbil obligó a que los perros fueran abandonados por sus dueños; décadas después, estos formaron grupos genéticos distintos",Ciencia
163,"Es falso que BTS haya sido captado durante un ensayo previo a sus conciertos en Perú: virales muestran a ENHYPEN, otro grupo surcoreano que visitó el país en julio",Verificador
152,"Producción de la película 'Santidad' denuncia censura y maltrato de los cines: ""Dicen que no debería tergiversar la buena fe o la moral en su contenido""",Cine y series
151,"Cantante argentino Adrián Barilari, vocalista de Rata Blanca, celebra triunfo de su imitador Nikólas Bujahico en la final de 'Yo soy': ""Felicitaciones""",Espectáculos


largo_titular,titular,seccion_original
18,Lo mejor de Molina,Molina
29,El gobierno de la distracción,Opinión
29,"Los años de Odría, José Ragas",Opinión
33,El Estado de Derecho no está solo,Opinión
33,El fin de la coartada fujimorista,Opinión


## 6. Exportación

In [55]:
COLUMNAS_FINALES = COLUMNAS + ["etiqueta"]
CATEGORIAS = {"politica", "economia", "deportes", "ciencia_tecnologia"}

df = noticias[COLUMNAS_FINALES]  # tabla limpia de la sección 6, sin la columna de control `fuente`
assert df.shape[1] == 7, f"Columnas inesperadas: {list(df.columns)}"
assert df["url"].is_unique, "Hay URL repetidas"
assert df["titular"].str.strip().ne("").all(), "Hay titulares vacíos"
assert df["etiqueta"].dropna().isin(CATEGORIAS).all(), "Hay etiquetas fuera de las 4 categorías"

destino = Path("datos") / "noticias.csv"
df.to_csv(destino, index=False, encoding="utf-8-sig")
assert pd.read_csv(destino).shape == df.shape, "El archivo guardado no coincide con el DataFrame"

print(f"Guardado: {destino}")
print("df.shape:", df.shape)
df.head(5)

Guardado: datos\noticias.csv
df.shape: (351, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,John Kelvin llora y pide ver a sus hijos ante ...,El cantante John Kelvin se dirigió públicament...,La República,Espectáculos,https://larepublica.pe/espectaculos/2026/10/18...,2026-10-18T22:37:33-05:00,NaN
1,UTC jugará la Liga 2 en 2027: es el primer des...,UTC perdió 1-0 frente a Cienciano y se convirt...,La República,Deportes,https://larepublica.pe/deportes/2026/10/18/utc...,2026-10-18T22:31:59-05:00,deportes
2,Caso Jhon Leiva: Dictan 9 meses de prisión pre...,La Fiscalía sostiene que Freider A. habría pro...,La República,Sociedad,https://larepublica.pe/sociedad/2026/10/18/cas...,2026-10-18T21:56:39-05:00,NaN
3,"Jorge Nieto a Álvaro Vargas Llosa: ""Su rol de ...",El líder de Buen Gobierno resaltó que su parti...,La República,Política,https://larepublica.pe/politica/2026/10/18/jor...,2026-10-18T21:50:00-05:00,politica
4,Una resistencia olvidada frenó la expansión de...,Las comunidades antiguas vivían en pequeños as...,La República,Ciencia,https://larepublica.pe/ciencia/2026/10/18/una-...,2026-10-18T21:33:09-05:00,ciencia_tecnologia


## 7. Documentación de bloqueos

| Fuente | ¿La permite robots.txt? | Qué pasó | Uso |
|---|---|---|---|
| `/ultimas-noticias` (HTML) | Sí | Responde `200`, pero solo trae 30 noticias | Fuente 1 |
| `/ultimas-noticias/pag/N` | **No** (`Disallow: /*/pag/`) | No se pide | Bloqueo resuelto con RSS |
| `/rss/*.xml` (RSS oficiales) | Sí (`Allow: /rss/*`) | ~200 noticias únicas de las últimas ~48 h | Fuente 2 |
| Portadas de sección (`/politica`, …) | Sí | Entre 7 y 65 noticias por portada | Fuente 3, solo hasta llegar a 350 |
| `/buscador`, `/archivo/*`, `/*/tag/*` | No | No se piden | — |
